# Simple baselines + format checker + scorer

A quick walk-through of the **majority** and **random** baselines for both subtasks, followed by the **format checker** and the **scorer**. These are the fastest way to produce a valid submission file and to verify the end-to-end pipeline.

**Prerequisite:** `python data/download_data.py` has been run.

In [ ]:
import sys, subprocess
from pathlib import Path

TASK1 = Path.cwd().resolve()
while TASK1.name != 'task1' and TASK1.parent != TASK1:
    TASK1 = TASK1.parent
DATA = TASK1 / 'data' / 'splits'
PRED = TASK1 / 'predictions'
PRED.mkdir(exist_ok=True)

def sh(*args):
    print('$', ' '.join(args))
    print(subprocess.check_output([sys.executable, *args], text=True, cwd=TASK1))

## 1. Majority baseline

Predicts the most-frequent class from the training set on every record.

* Subtask A1: predicts `Not Hateful` (the larger class) for every meme.
* Subtask A2: predicts the single most-frequent fine-grained sub-type (`Sarcasm` on the released data) for every meme.

In [ ]:
sh('baselines/majority_baseline.py', '--subtask', 'a1',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'majority_a1.tsv'),
   '--run-id', 'majority')

sh('baselines/majority_baseline.py', '--subtask', 'a2',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'majority_a2.jsonl'))

## 2. Random baseline

Bernoulli draw with training-set priors (A1) or independent multi-label draws with per-class priors (A2). Useful as a sanity-check ceiling for trivial systems.

In [ ]:
sh('baselines/random_baseline.py', '--subtask', 'a1',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'random_a1.tsv'),
   '--run-id', 'random', '--seed', '42')

sh('baselines/random_baseline.py', '--subtask', 'a2',
   '--train', str(DATA / 'train.jsonl'),
   '--target', str(DATA / 'dev_test.jsonl'),
   '--out', str(PRED / 'random_a2.jsonl'), '--seed', '42')

## 3. Format check

Always run the format checker before submitting.

In [ ]:
for subtask, ext in [('a1', 'tsv'), ('a2', 'jsonl')]:
    for kind in ('majority', 'random'):
        sh('format_checker/format_checker.py',
           '--subtask', subtask,
           '--predictions', str(PRED / f'{kind}_{subtask}.{ext}'))

## 4. Local scoring against `dev.jsonl`

The dev split is labelled, so we can compute metrics on it locally. (For `dev_test` the leaderboard is the only source of metrics during the development phase.)

We re-run the baselines with `--target dev.jsonl` to align the prediction IDs with the gold IDs.

In [ ]:
sh('baselines/majority_baseline.py', '--subtask', 'a1',
   '--train', str(DATA / 'train.jsonl'), '--target', str(DATA / 'dev.jsonl'),
   '--out', str(PRED / 'majority_a1_dev.tsv'), '--run-id', 'majority_dev')

sh('scorer/scorer.py', '--subtask', 'a1',
   '--gold', str(DATA / 'dev.jsonl'),
   '--predictions', str(PRED / 'majority_a1_dev.tsv'))

Same for Subtask A2:

In [ ]:
sh('baselines/majority_baseline.py', '--subtask', 'a2',
   '--train', str(DATA / 'train.jsonl'), '--target', str(DATA / 'dev.jsonl'),
   '--out', str(PRED / 'majority_a2_dev.jsonl'))
sh('scorer/scorer.py', '--subtask', 'a2',
   '--gold', str(DATA / 'dev.jsonl'),
   '--predictions', str(PRED / 'majority_a2_dev.jsonl'))

Your baselines should beat these majority numbers \u2014 if not, something is wrong with your training loop.